# Handling Data for Nepali Post Sentiment Analysis

Following datasets will be used for sentiment analysis of emotions in different posts online. 
- https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
- https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv
- https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_3.csv

This dataset is choosen to let model understand different emotion via texts. This will allow us to get information on emotion or sentiment of human writing a post. Also the data from reddit where Nepali people posts will be collected and categorized and analysed using KMeans then again feed to the senitment analysis to get overall picture of nepali people's discussion and their emotions regarding it.


## Download GoEmotions dataset

### import libraries 
- request
- pandas

In [1]:
import requests
import pandas as pd 
import os

In [2]:
go_emotion_base_url = "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/"

# go emotion datasets names
go_emotion_datasets_names = [
    "goemotions_1.csv",
    "goemotions_2.csv",
    "goemotions_3.csv"
]

# make an empty list to append loaded pd
go_emotions_list = []

# use loop for goemotion
for goemotion in go_emotion_datasets_names:
    df = pd.read_csv(f"{go_emotion_base_url}{goemotion}")
    go_emotions_list.append(df)
    print(f"{goemotion} Loaded!")
    
go_emotions_dataset = pd.concat(go_emotions_list, ignore_index=True)

print(f"SHAPE OF FINAL GO_EMOTION DATASET = {go_emotions_dataset.shape}")
go_emotions_dataset.head(3)

goemotions_1.csv Loaded!
goemotions_2.csv Loaded!
goemotions_3.csv Loaded!
SHAPE OF FINAL GO_EMOTION DATASET = (211225, 37)


,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1


## Clean GoEmotions Data as per Needed:
- remove id, author, subreddit, link_id, parent_id, created_utc and rater_id, example_very_unclear
- clean text column (text preprocessing)

In [3]:
# drop unnecessary columns
go_emotions_dataset.drop(columns=[
    "id", "author", "subreddit", "link_id", 
    "parent_id", "created_utc", "rater_id", 
    "example_very_unclear"
], inplace=True)

print("Dropped unnecessary columns in place!")

Dropped unnecessary columns in place!


### Clean text requires additional libraries:
- regex
- bs4 BeautifulSoup
- string

Create a function to clean text usable in future as well. This function will remove html, punctuations, urls, additonal spaces, emojis, and also lowercase the text. It is required as a basic pre-processing for TF-IDF NLP.

In [4]:
import re
from bs4 import BeautifulSoup
import emoji

REMOVE_PHRASES = [
    "nice video",
    "great video",
    "awesome video",
    "amazing video",
    "excellent video",
    "good video",
    "best video",
    "nice content",
    "great content",
    "awesome content",
    "good content",
    "keep it up",
    "good job",
    "great job",
    "awesome job",
    "well done",
    "thanks for sharing",
    "thank you for sharing",
    "thanks for the video",
    "thank you for the video",
    "thanks for uploading",
    "more videos please",
    "upload more",
    "keep uploading",
    "keep making videos",
    "first comment",
    "early gang",
    "still watching",
    "watching in 2026",
    "who is watching in 2026",
    "anyone here in 2026"
]

REMOVE_WORDS = {
    "thanks",
    "thank",
    "video",
    "videos",
    "content",
    "podcast",
    "bro",
    "sir",
    "nice",
    "great",
    "awesome",
    "amazing",
    "good",
    "best",
    "wow",
    "pls",
    "please"
}


def clean_text(text):

    if not isinstance(text, str):
        return ""

    # Remove HTML
    text = BeautifulSoup(text, "html.parser").get_text()

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove emojis
    text = emoji.replace_emoji(text, "")

    # Remove punctuation (keep English + Devanagari)
    text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text)

    # Remove common useless phrases
    for phrase in REMOVE_PHRASES:
        text = text.replace(phrase, " ")

    # Remove common useless words
    words = [
        word for word in text.split()
        if word not in REMOVE_WORDS
    ]

    # Remove extra spaces
    text = " ".join(words)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
# Apply the function to text column
go_emotions_dataset["text"] = go_emotions_dataset["text"].apply(clean_text)

# See few texts how  they look
go_emotions_dataset.head()

,text,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,that game hurt,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,sexuality shouldn t be a grouping category it ...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,you do right if you don t care then fuck em,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,man i love reddit,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,name was nowhere near them he was by the falcon,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


### Drop null values if any and Save

In [6]:
# Drop na value and save
go_emotions_dataset = (
    go_emotions_dataset
    .dropna(subset=["text"])
)
# Drop empty string
go_emotions_dataset = go_emotions_dataset[
    go_emotions_dataset["text"].str.strip() != ""
].reset_index(drop=True)

In [7]:
go_emotions_dataset = go_emotions_dataset.sample(frac=1, random_state=41)

# Save to the datasets folder
go_emotions_dataset.to_csv("./datasets/go_emotions.csv", index=False)

## Download YT Comments
This data will be downloaded googles Youtube API key. This dataset will contain nepali's voice including youths. The comments represent what nepali people think on current government, system, justice, education, etc. This shows if citizens are happy or not in different aspects of the country.

### Import Libraries
- build from googleapiclient.discovery
- dotenv
- LangDetectException

In [8]:
from dotenv import load_dotenv
from googleapiclient.discovery import build

# config dotenv
load_dotenv("./.env")

True

In [9]:
# youtube api key access
youtube = build(
    "youtube",
    "v3",
    developerKey=os.getenv("YT_API_KEY")
)

### Extracts channel ID
The channel id is not usually present like it used to, so it is necessary to extract the channel's id before extracting contents from the Api.

In [10]:
def get_channel_id(channel):

    # Already a channel ID
    if channel.startswith("UC"):
        return channel


    # Handle (@name)
    if channel.startswith("@"):

        response = youtube.channels().list(
            part="id",
            forHandle=channel.replace("@","")
        ).execute()


        if not response.get("items"):
            raise Exception(
                f"Channel not found: {channel}"
            )

        return response["items"][0]["id"]

    raise Exception(
        f"Invalid channel format: {channel}"
    )

### Selection of Channels
Only few channels are selected:
- TechTanka
- WhySoOffend
- Thaharesearch
- eon_visuals
- TheNepaliComment

These channels are popular in the youtube explaining current problems and news of Nepal. They also talk about: politics, corruption, system, justice, etc which brings engaging viewrs on the video who comment and share their point of view.

In [11]:
channels = [
    # News
    get_channel_id("@RoutineofNepalBanda"),
    get_channel_id("@SidhaKura_"),
    get_channel_id("@insidenepalnews"),
    get_channel_id("@samacharpati"),

    # Politics
    get_channel_id("@WhySoOffended"),
    get_channel_id("@TheNepaliComment"),
    get_channel_id("@MisguidedNepal"),
    get_channel_id("@inside-kura"),

    # Podcasts
    get_channel_id("@Sushant_Pradhan"),
    get_channel_id("@hernekatha"),

    # Research
    get_channel_id("@Thaharesearch"),
    get_channel_id("@IdeapreneurNepal"),

    # Tech
    get_channel_id("@TechTanka"),
    get_channel_id("@GadgetByteNepali"),
    get_channel_id("@PratimaAdhikariTech"),

    # Economy
    get_channel_id("@merolaganiofficial"),
    get_channel_id("@shareSansar2641"),

    # Entertainment
    get_channel_id("@OSRDigital"),
    get_channel_id("@budhasubbamusic"),
    get_channel_id("@highlightsnepal2009"),

    # Sports
    get_channel_id("@HamroKhelkud"),

    # Documentary
    get_channel_id("@eon_visuals"),
    get_channel_id("@WhatsWith."),

    # Youth
    get_channel_id("@SwagatGyawali1"),
    get_channel_id("@NepalShowofficial"),

    # General
    get_channel_id("@BishwoGhatana9")
]

### Extract playlists and recent videos

In [12]:
def get_upload_playlist(channel_id):

    response = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    ).execute()
    return response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

In [13]:
def get_recent_videos(
    channel_id,
    start_date,
    end_date,
    limit=10
):
    response = youtube.search().list(
        part="snippet",
        channelId=channel_id,
        order="date",
        maxResults=limit,
        type="video",
        publishedAfter=start_date,
        publishedBefore=end_date
    ).execute()

    videos=[]

    for item in response.get("items", []):

        videos.append({
            "video_id": item["id"]["videoId"],
            "title": item["snippet"]["title"],
            "date": item["snippet"]["publishedAt"]
        })

    return videos

### Recieve Comments
This function is the main function used internally to fetch all the comments in the video. 

In [14]:
def get_comments(video_id, limit=500):

    comments=[]
    
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )

    while request and len(comments) < limit:
        try:
            response = request.execute()
        except:
            break

        for item in response["items"]:
            snippet = (
                item["snippet"]
                ["topLevelComment"]
                ["snippet"]
            )

            comments.append({
                "comment": snippet["textDisplay"],
                "published_at": snippet["publishedAt"]
            })

        token = response.get("nextPageToken")

        if token:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                pageToken=token,
                maxResults=100,
                textFormat="plainText"
            )
        else:
            break
        
    return comments

### Quality control of Comments
This function only passes the quality comments. The quality control must fullfill following criteria for passing:
- Language must be English (NO-Devnagari)
- Length of raw text must be above 5
- Remove low value adding phrases

No-devnagari because devnagari will add additional challange to preprocess, meaning extraction and also creates larger groups for model's embedding to understand. Also there are limitations on semantic datasets for devnagari (NEPALI). This will also affect K-Means Clustering as the model might just seperate Devnagari embedding to a seperate chunk not giving approprite meaning with english embeddings.

Length of text must be above 5 because raw text too short might give lower semantic value. This does not give detail emotion on how person relates and feels, a simple " I have done this ..... and ..... which caused me to be depressed." structured is needed instead of "I am very sad." For model to generalize well.

Removal of useless phrases simply means removing comments that contains low value adding texts such has : nice video, wow, upload more, etc. as this does not align with the project's problem and rather focus on what people are talking about video not problem.

In [15]:
def is_quality_comment(text):
    if len(text.split()) < 10:
        return False

    low=text.lower().strip()

    useless = [
    "best podcast",
    "best video",
    "video",
    "nice video",
    "great video",
    "awesome video",
    "amazing video",
    "excellent video",
    "good video",
    "very good video",
    "best video",
    "cool video",
    "beautiful video",
    "wonderful video",
    "fantastic video",
    "brilliant video",
    "incredible video",
    "interesting video",
    "nice content",
    "great content",
    "awesome content",
    "amazing content",
    "good content",
    "best content",
    "love this content",
    "loved this content",
    "keep it up",
    "thanks for the video",
    "thank you for the video",
    "thanks for uploading",
    "thanks for sharing",
    "thank you for sharing",
    "thanks for making this",
    "appreciate the video",
    "appreciate this video",
    "nice one",
    "good one",
    "great one",
    "awesome one",
    "well done",
    "good job",
    "great job",
    "awesome job",
    "first",
    "first comment",
    "early",
    "early gang",
    "who is watching in 2026",
    "anyone here in 2026",
    "still watching",
    "watching in 2026",
    "make more videos",
    "good morning",
    "good afternoon",
    "goog night",
    "more videos please",
    "upload more",
    "keep uploading",
    "keep making videos",
    "keep it up",
    "thanks",
    "thank you"
]

    for phrase in useless:
        if phrase in low:
            return False

    return True

### The main Scrape Function
Extracts video -> Extracts Comment -> Checks Quality -> Use/Throw

In [16]:
def scrape_channels(channels, start_date, end_date, limit):

    data=[]

    for channel_id in channels:
        print("CHANNEL:", channel_id)
        videos=get_recent_videos(
            channel_id,
            start_date,
            end_date,
            limit
        )
        
        for video in videos:
            print(
                "VIDEO:",
                video["title"]
            )
            comments=get_comments(
                video["video_id"]
            )
            for comment in comments:
                if is_quality_comment(comment["comment"]):
                    data.append({
                        "channel":channel_id,
                        "video":video["title"],
                        "video_id":video["video_id"],
                        "comment":comment["comment"],
                        "published_at":comment["published_at"]
                    })

    return pd.DataFrame(data)

In [17]:
yt_comments = scrape_channels(channels, "2024-01-01T00:00:00Z", "2026-07-25T00:00:00Z", 10)
yt_comments.head()

CHANNEL: UC2rariKhGBaLf0qmhWQv3KQ
VIDEO: Nepal&#39;s First Modern Tunnel? l  RONB Explained
VIDEO: Is kichkandi Real ? l Highway ko Kichkandi l RONB Explained  #ronb
VIDEO: Is kichkandi Real ? l Highway ko Kichkandi l RONB Explained
VIDEO: Fast Track or Slow Track? | RONB Explained
VIDEO: FINAL MATCH (MANTAIN FC _CURUCAO vs INSIGHT VISION_ BRAZIL)  RONB FUTSAL 2026
VIDEO: RONB Futsal 2026 | Live 🔴 | Day 5 | FINAL DAY
VIDEO: 3rd PLACE ENGLAND (HIMALAYA FC) V JORDAN (KORIBATI FC) RONB FUTSAL 2026
VIDEO: SEMI FINAL 2- BRAZIL (INSIGHT VISION) V JORDAN (KORIBATI FC) RONB FUTSAL 2026
VIDEO: RONB Futsal 2026 | Live 🔴| Day 4
VIDEO: RONB Futsal 2026 | Live 🔴| Day 4
CHANNEL: UC8CTPZVjEowxUO4dqRpyksA
VIDEO: गृहमन्त्री सुधन गुरुङ LIVE, संघीयता सबलीकरण तथा राष्ट्रिय सरोकार समितिको बैठक
VIDEO: नेपालमै पहिलो पटक गाँजा खेतिलाई वैधानिकता  || Cannabis legalized in Nepal !  || SIDHAKURA ||
VIDEO: काँग्रेसको जरा अभियानको समापन र नयाँ अभियानको घोषणा LIVE
VIDEO: काँग्रेसको जरा अभियानको समापन र नयाँ अभियानको

,channel,video,video_id,comment,published_at
0,UC2rariKhGBaLf0qmhWQv3KQ,Nepal&#39;s First Modern Tunnel? l RONB Expla...,XnPOuwslU5o,Aba Balen ley surung marga ma chai k nei garer...,2026-07-26T02:29:19Z
1,UC2rariKhGBaLf0qmhWQv3KQ,Nepal&#39;s First Modern Tunnel? l RONB Expla...,XnPOuwslU5o,Dai affno sathi ko pani kae galati oulauni hok...,2026-07-25T18:45:31Z
2,UC2rariKhGBaLf0qmhWQv3KQ,Nepal&#39;s First Modern Tunnel? l RONB Expla...,XnPOuwslU5o,So sad to hear this why they not content Nepal...,2026-07-25T16:57:04Z
3,UC2rariKhGBaLf0qmhWQv3KQ,Nepal&#39;s First Modern Tunnel? l RONB Expla...,XnPOuwslU5o,केपि oli को vision ले गर्दा यो सम्भव भएको हो। ...,2026-07-25T09:44:08Z
4,UC2rariKhGBaLf0qmhWQv3KQ,Nepal&#39;s First Modern Tunnel? l RONB Expla...,XnPOuwslU5o,कमसेकम थम्बनेलमा त केपी को फोटो राखओ ए गँजेडी ...,2026-07-25T04:44:40Z


## Mark Devnagari, Roman or english

In [19]:
import re
from wordfreq import zipf_frequency

# Detect Devanagari Unicode
DEVANAGARI_PATTERN = re.compile(r'[\u0900-\u097F]')

def detect_language(text):
    text = str(text).strip()

    if not text:
        return "unknown"

    # 1. Devanagari Nepali
    if DEVANAGARI_PATTERN.search(text):
        return "devanagari_nepali"

    # Extract alphabetic words
    words = re.findall(r"[a-zA-Z']+", text.lower())

    if len(words) == 0:
        return "unknown"

    # Count how many words look like English
    english_count = sum(
        zipf_frequency(word, "en") > 3
        for word in words
    )

    ratio = english_count / len(words)

    # Mostly English words
    if ratio >= 0.7:
        return "english"

    # Otherwise assume Roman Nepali
    return "roman_nepali"

In [20]:
yt_comments["clean_comment"] = yt_comments["comment"].apply(clean_text)
# Remove with low length
yt_comments = yt_comments[
    yt_comments["clean_comment"].str.split().str.len() >= 8
].reset_index(drop=True)# remove nan vals and save
yt_comments = yt_comments.dropna()
yt_comments = yt_comments[yt_comments["comment"]!=""]
# label languate
yt_comments["language"] = yt_comments["comment"].apply(detect_language)
# random state shuffle
yt_comments = yt_comments.sample(frac=1, random_state=41)

# show data
print(f"Total Length of data = {yt_comments.shape}")
yt_comments.head()

Total Length of data = (9136, 7)


,channel,video,video_id,comment,published_at,clean_comment,language
3272,UCZuKpNyipL9W-OkKwCMwVNQ,हावामा के छ ? - भाग १ | What is in the Air? | ...,R1c27P_gG5k,Main concern of this Time Adverse effects of ...,2026-06-05T16:49:08Z,main concern of this time adverse effects of i...,english
5192,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,बालेनको स्थानीय सरकारको भूमिका प्रशंसनीय रह्यो...,2026-07-19T04:58:31Z,बालेनको स्थानीय सरकारको भूमिका प्रशंसनीय रह्यो...,devanagari_nepali
579,UC2CAM55AnDflrpC20uH700A,Nepali OTT&#39;s Biggest Bet Ever | WSO | Bina...,-XN_V4ebOGc,Money making tournament bhayera ta fifa presi...,2026-07-18T07:05:13Z,money making tournament bhayera ta fifa presid...,roman_nepali
4177,UCZuKpNyipL9W-OkKwCMwVNQ,हेटौंडा कपडा उद्योगको कथा | Hetauda Kapada Ud...,7PgE0tLgSF8,हेर मेरो देस को बिजोक अब चल्न्नु पर छ ।।।,2026-04-12T16:42:26Z,हेर मेरो देस को बिजोक अब चल्न्नु पर छ ।।।,devanagari_nepali
3496,UCZuKpNyipL9W-OkKwCMwVNQ,पिएचडी किसानको कथा | Story of a PHD Farmer | ...,38NauB4dMec,चकलाबंदी गरे र मात्र देश को खेती र खाद्यान्न म...,2026-05-23T07:29:11Z,चकलाबंदी गरे र मात्र देश को खेती र खाद्यान्न म...,devanagari_nepali


In [21]:
# youtube comments saved
yt_comments.to_csv("./datasets/yt_comments.csv", index=False)
